# Regression Algorithms for Beginners

## What You'll Learn

Regression is used to predict **continuous numerical values**. In this notebook, you'll master:

1. **Linear Regression** - The foundation of regression
2. **Polynomial Regression** - For non-linear relationships
3. **Ridge Regression (L2)** - Regularization to prevent overfitting
4. **Lasso Regression (L1)** - Feature selection through regularization
5. **Decision Tree Regression** - Tree-based approach
6. **Random Forest Regression** - Ensemble of trees

**Real-World Applications:**
- House price prediction
- Stock price forecasting
- Sales prediction
- Temperature forecasting

---

## Setup: Import Libraries and Load Data

In [ ]:
# Install required packages (uncomment if needed)
# !pip install numpy pandas matplotlib seaborn scikit-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

## Load Dataset: California Housing

We'll use the **California Housing dataset** - a classic regression dataset for predicting house prices.

**Features:**
- MedInc: Median income in block
- HouseAge: Median house age in block
- AveRooms: Average number of rooms
- AveBedrms: Average number of bedrooms
- Population: Block population
- AveOccup: Average house occupancy
- Latitude: Block latitude
- Longitude: Block longitude

**Target:** MedHouseVal (Median house value in $100,000s)

In [ ]:
from sklearn.datasets import fetch_california_housing

# Load dataset
housing = fetch_california_housing()

# Create DataFrame
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

print(f"Dataset Shape: {df.shape}")
print(f"\nFeatures: {list(housing.feature_names)}")
print(f"\nTarget: MedHouseVal (Median House Value in $100,000s)")
df.head()

## Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Distribution of target variable
plt.figure(figsize=(10, 6))
plt.hist(df['MedHouseVal'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Median House Value ($100,000s)')
plt.ylabel('Frequency')
plt.title('Distribution of House Prices')
plt.axvline(df['MedHouseVal'].mean(), color='red', linestyle='--', label=f'Mean: ${df["MedHouseVal"].mean()*100000:,.0f}')
plt.legend()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
correlation = df.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

# Show correlations with target
print("\nCorrelation with House Price:")
print(correlation['MedHouseVal'].sort_values(ascending=False))

In [ ]:
# Scatter plots of top features vs target
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# MedInc vs Price
axes[0, 0].scatter(df['MedInc'], df['MedHouseVal'], alpha=0.3)
axes[0, 0].set_xlabel('Median Income')
axes[0, 0].set_ylabel('House Price ($100k)')
axes[0, 0].set_title('Income vs House Price')

# HouseAge vs Price
axes[0, 1].scatter(df['HouseAge'], df['MedHouseVal'], alpha=0.3)
axes[0, 1].set_xlabel('House Age')
axes[0, 1].set_ylabel('House Price ($100k)')
axes[0, 1].set_title('House Age vs House Price')

# AveRooms vs Price
axes[1, 0].scatter(df['AveRooms'], df['MedHouseVal'], alpha=0.3)
axes[1, 0].set_xlabel('Average Rooms')
axes[1, 0].set_ylabel('House Price ($100k)')
axes[1, 0].set_title('Rooms vs House Price')
axes[1, 0].set_xlim(0, 15)  # Limit outliers

# Geographic plot
scatter = axes[1, 1].scatter(df['Longitude'], df['Latitude'], 
                             c=df['MedHouseVal'], cmap='viridis', alpha=0.3, s=5)
axes[1, 1].set_xlabel('Longitude')
axes[1, 1].set_ylabel('Latitude')
axes[1, 1].set_title('Geographic Distribution of House Prices')
plt.colorbar(scatter, ax=axes[1, 1], label='Price ($100k)')

plt.tight_layout()
plt.show()

## Data Preparation

In [ ]:
# Prepare features and target
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Scale features for better performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

## Evaluation Metrics for Regression

Before we build models, let's understand how we'll evaluate them:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** (Mean Absolute Error) | avg(|actual - predicted|) | Average error in same units as target |
| **MSE** (Mean Squared Error) | avg((actual - predicted)^2) | Penalizes large errors more |
| **RMSE** (Root MSE) | sqrt(MSE) | Same units as target |
| **R² Score** | 1 - (SS_res/SS_tot) | % of variance explained (0-1 is good) |

In [ ]:
# Helper function to evaluate models
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train model and return evaluation metrics
    """
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    results = {
        'Model': model_name,
        'Train R²': r2_score(y_train, y_train_pred),
        'Test R²': r2_score(y_test, y_test_pred),
        'Test MAE': mean_absolute_error(y_test, y_test_pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred))
    }
    
    return results, y_test_pred

---

# Algorithm 1: Linear Regression

## The Foundation of Regression

**Concept:** Find the best straight line that fits the data.

**Formula:** y = b₀ + b₁x₁ + b₂x₂ + ... + bₙxₙ

Where:
- y = predicted value
- b₀ = intercept (y when all x = 0)
- bᵢ = coefficient for feature xᵢ

**Goal:** Minimize the sum of squared errors between predictions and actual values.

In [ ]:
# Simple visualization of Linear Regression concept
# Using just MedInc (most correlated feature)

plt.figure(figsize=(10, 6))

# Get a sample for visualization
sample_idx = np.random.choice(len(df), 1000, replace=False)
X_sample = df['MedInc'].iloc[sample_idx].values.reshape(-1, 1)
y_sample = df['MedHouseVal'].iloc[sample_idx].values

# Fit simple linear regression
simple_lr = LinearRegression()
simple_lr.fit(X_sample, y_sample)

# Plot
plt.scatter(X_sample, y_sample, alpha=0.5, label='Data points')
X_line = np.linspace(X_sample.min(), X_sample.max(), 100).reshape(-1, 1)
plt.plot(X_line, simple_lr.predict(X_line), color='red', linewidth=2, label='Best fit line')
plt.xlabel('Median Income')
plt.ylabel('House Price ($100k)')
plt.title('Linear Regression: Finding the Best Fit Line')
plt.legend()
plt.show()

print(f"Equation: Price = {simple_lr.intercept_:.2f} + {simple_lr.coef_[0]:.2f} × Income")

In [ ]:
# Full Linear Regression with all features
lr_model = LinearRegression()
lr_results, lr_pred = evaluate_model(
    lr_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Linear Regression'
)

print("=" * 50)
print("LINEAR REGRESSION RESULTS")
print("=" * 50)
for key, value in lr_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Feature importance (coefficients)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
plt.xlabel('Coefficient Value')
plt.title('Linear Regression: Feature Coefficients')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.show()

print("\nFeature Importance (by coefficient magnitude):")
print(feature_importance)

---

# Algorithm 2: Polynomial Regression

## Capturing Non-Linear Relationships

**Concept:** Transform features to capture curved relationships.

**Example (degree=2):**
- Original: [x₁, x₂]
- Transformed: [x₁, x₂, x₁², x₁x₂, x₂²]

**When to use:** When scatter plots show curved patterns.

In [ ]:
# Visualize polynomial regression concept
np.random.seed(42)
X_demo = np.linspace(0, 10, 100).reshape(-1, 1)
y_demo = 3 + 2*X_demo.ravel() - 0.5*X_demo.ravel()**2 + np.random.normal(0, 2, 100)

plt.figure(figsize=(14, 5))

for i, degree in enumerate([1, 2, 5]):
    plt.subplot(1, 3, i+1)
    
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree)
    X_poly = poly.fit_transform(X_demo)
    
    # Fit model
    model = LinearRegression()
    model.fit(X_poly, y_demo)
    y_pred_demo = model.predict(X_poly)
    
    # Plot
    plt.scatter(X_demo, y_demo, alpha=0.5, label='Data')
    plt.plot(X_demo, y_pred_demo, color='red', linewidth=2, label=f'Degree {degree}')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title(f'Polynomial Degree {degree}')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Polynomial Regression on housing data (degree=2)
# Note: Higher degrees can cause overfitting!

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Original features: {X_train_scaled.shape[1]}")
print(f"Polynomial features: {X_train_poly.shape[1]}")

In [ ]:
# Train polynomial regression
poly_model = LinearRegression()
poly_results, poly_pred = evaluate_model(
    poly_model, X_train_poly, X_test_poly, y_train, y_test, 'Polynomial Regression (d=2)'
)

print("=" * 50)
print("POLYNOMIAL REGRESSION RESULTS")
print("=" * 50)
for key, value in poly_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

---

# Algorithm 3: Ridge Regression (L2 Regularization)

## Preventing Overfitting

**Problem:** Linear regression can overfit when:
- Many features
- Correlated features
- High polynomial degrees

**Solution:** Add a penalty for large coefficients.

**Ridge Formula:** Minimize (MSE + α × Σbᵢ²)

Where α (alpha) controls regularization strength:
- Higher α = simpler model (smaller coefficients)
- Lower α = more complex model (closer to linear regression)

In [ ]:
# Ridge Regression with different alpha values
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_results_list = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    y_pred = ridge.predict(X_test_scaled)
    
    ridge_results_list.append({
        'Alpha': alpha,
        'Test R²': r2_score(y_test, y_pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    })

ridge_comparison = pd.DataFrame(ridge_results_list)
print("Ridge Regression Results with Different Alpha Values:")
print(ridge_comparison)

In [ ]:
# Best Ridge model
ridge_model = Ridge(alpha=1.0)
ridge_results, ridge_pred = evaluate_model(
    ridge_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Ridge Regression'
)

print("=" * 50)
print("RIDGE REGRESSION RESULTS (alpha=1.0)")
print("=" * 50)
for key, value in ridge_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Visualize coefficient shrinkage
plt.figure(figsize=(12, 6))

alphas_viz = np.logspace(-2, 4, 100)
coefs = []

for alpha in alphas_viz:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    coefs.append(ridge.coef_)

coefs = np.array(coefs)

for i, feature in enumerate(X.columns):
    plt.plot(alphas_viz, coefs[:, i], label=feature)

plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)')
plt.ylabel('Coefficient Value')
plt.title('Ridge: Coefficient Shrinkage as Alpha Increases')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---

# Algorithm 4: Lasso Regression (L1 Regularization)

## Feature Selection Through Regularization

**Lasso Formula:** Minimize (MSE + α × Σ|bᵢ|)

**Key Difference from Ridge:**
- Lasso can set coefficients **exactly to zero**
- Acts as automatic **feature selection**
- Useful when you suspect only a few features matter

In [ ]:
# Lasso Regression
lasso_model = Lasso(alpha=0.01)
lasso_results, lasso_pred = evaluate_model(
    lasso_model, X_train_scaled, X_test_scaled, y_train, y_test, 'Lasso Regression'
)

print("=" * 50)
print("LASSO REGRESSION RESULTS")
print("=" * 50)
for key, value in lasso_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Compare coefficients: Linear vs Ridge vs Lasso
comparison_df = pd.DataFrame({
    'Feature': X.columns,
    'Linear': lr_model.coef_,
    'Ridge': ridge_model.coef_,
    'Lasso': lasso_model.coef_
})

print("Coefficient Comparison:")
print(comparison_df)

# Visualize
comparison_df.set_index('Feature').plot(kind='bar', figsize=(12, 6))
plt.title('Coefficient Comparison: Linear vs Ridge vs Lasso')
plt.ylabel('Coefficient Value')
plt.xticks(rotation=45)
plt.legend(title='Model')
plt.tight_layout()
plt.show()

---

# Algorithm 5: Decision Tree Regression

## Tree-Based Learning

**Concept:** Make predictions by learning decision rules from features.

**How it works:**
1. Split data based on feature thresholds
2. Create branches for each split
3. Repeat until stopping criteria
4. Predict: average of training samples in each leaf

**Advantages:**
- No scaling needed
- Handles non-linear relationships
- Easy to interpret

**Disadvantages:**
- Prone to overfitting
- Sensitive to data changes

In [ ]:
# Decision Tree with different max_depth values
depths = [3, 5, 10, None]  # None means no limit
dt_results_list = []

for depth in depths:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)  # Note: using unscaled data
    
    y_train_pred = dt.predict(X_train)
    y_test_pred = dt.predict(X_test)
    
    dt_results_list.append({
        'Max Depth': depth if depth else 'None',
        'Train R²': r2_score(y_train, y_train_pred),
        'Test R²': r2_score(y_test, y_test_pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_test_pred))
    })

dt_comparison = pd.DataFrame(dt_results_list)
print("Decision Tree Results with Different Depths:")
print(dt_comparison)
print("\nNote: Large gap between Train and Test R² indicates overfitting!")

In [ ]:
# Best Decision Tree model (balanced depth)
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_results, dt_pred = evaluate_model(
    dt_model, X_train, X_test, y_train, y_test, 'Decision Tree'
)

print("=" * 50)
print("DECISION TREE RESULTS (max_depth=10)")
print("=" * 50)
for key, value in dt_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Feature importance from Decision Tree
dt_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(dt_importance['Feature'], dt_importance['Importance'])
plt.xlabel('Feature Importance')
plt.title('Decision Tree: Feature Importance')
plt.show()

---

# Algorithm 6: Random Forest Regression

## Ensemble of Trees

**Concept:** Combine many decision trees to reduce overfitting.

**How it works:**
1. Create multiple decision trees on random data subsets
2. Each tree uses random feature subsets
3. Final prediction = average of all trees

**Advantages:**
- Reduces overfitting
- Handles non-linearity
- Feature importance
- Robust to outliers

In [ ]:
# Random Forest Regression
rf_model = RandomForestRegressor(
    n_estimators=100,      # Number of trees
    max_depth=15,          # Maximum depth of each tree
    min_samples_split=5,   # Minimum samples to split
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)

rf_results, rf_pred = evaluate_model(
    rf_model, X_train, X_test, y_train, y_test, 'Random Forest'
)

print("=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
for key, value in rf_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Random Forest feature importance
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(rf_importance['Feature'], rf_importance['Importance'], color='forestgreen')
plt.xlabel('Feature Importance')
plt.title('Random Forest: Feature Importance')
plt.show()

---

# Model Comparison Summary

In [ ]:
# Compile all results
all_results = pd.DataFrame([
    lr_results,
    poly_results,
    ridge_results,
    lasso_results,
    dt_results,
    rf_results
])

print("=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
print(all_results.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² Score comparison
models = all_results['Model']
x_pos = np.arange(len(models))

axes[0].bar(x_pos - 0.2, all_results['Train R²'], 0.4, label='Train R²', color='steelblue')
axes[0].bar(x_pos + 0.2, all_results['Test R²'], 0.4, label='Test R²', color='coral')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(models, rotation=45, ha='right')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Score Comparison')
axes[0].legend()
axes[0].set_ylim(0, 1)

# RMSE comparison
axes[1].bar(models, all_results['Test RMSE'], color='teal')
axes[1].set_xticklabels(models, rotation=45, ha='right')
axes[1].set_ylabel('RMSE ($100k)')
axes[1].set_title('Test RMSE Comparison (Lower is Better)')

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted plot for best model
plt.figure(figsize=(10, 8))
plt.scatter(y_test, rf_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual House Price ($100k)')
plt.ylabel('Predicted House Price ($100k)')
plt.title('Random Forest: Actual vs Predicted')
plt.show()

---

## Key Takeaways

| Algorithm | Best For | Pros | Cons |
|-----------|----------|------|------|
| **Linear Regression** | Simple, linear relationships | Fast, interpretable | Assumes linearity |
| **Polynomial** | Non-linear patterns | Captures curves | Can overfit |
| **Ridge (L2)** | Many/correlated features | Prevents overfitting | No feature selection |
| **Lasso (L1)** | Feature selection needed | Automatic feature selection | May miss important features |
| **Decision Tree** | Non-linear, categorical | Easy to interpret | Overfits easily |
| **Random Forest** | General purpose | Robust, accurate | Slower, less interpretable |

---

## Practice Exercises

1. Load the Boston Housing dataset (or another regression dataset)
2. Perform EDA and identify important features
3. Compare at least 3 regression algorithms
4. Tune hyperparameters using cross-validation
5. Create actual vs predicted plots for each model

In [ ]:
# Practice: Try loading a different dataset
# Option 1: From Kaggle
# !kaggle datasets download -d camnugent/california-housing-prices

# Option 2: From HuggingFace
# from datasets import load_dataset
# dataset = load_dataset("scikit-learn/auto-mpg")

# Your code here...